In [ ]:
from skimage import io, filters, morphology, segmentation
from skimage.filters import median
from skimage.morphology import disk
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
filename = "drain_imbib-Stitched_final.tif"
# filename = "drain_imbib-Stitched_t0.tif"

currentDir = Path.cwd()
input_path = currentDir / filename
output_dir = currentDir / "threshold_results_2"
output_dir.mkdir(exist_ok=True)

In [ ]:
# Load grayscale image
image = io.imread(input_path, as_gray=True)

# 1. Reduce speckle without destroying edges too much
image_filt = median(image, disk(2))

# 2. Local threshold with moderate neighborhood
local_thresh = filters.threshold_local(
    image_filt,
    block_size=51,
    offset=2,
    method='gaussian'
)

# 3. Create mask
mask = image_filt > local_thresh

# 4. Clean mask
mask = morphology.remove_small_objects(mask, min_size=200)
mask = morphology.remove_small_holes(mask, area_threshold=200)

# 5. Extract thin interface
interface = segmentation.find_boundaries(mask, mode='inner')

In [ ]:
# Plot
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(image, cmap='gray')
ax[0].set_title('Original')
ax[1].imshow(mask, cmap='gray')
ax[1].set_title('Cleaned mask')
ax[2].imshow(interface, cmap='gray')
ax[2].set_title('Interface')
for a in ax:
    a.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
io.imsave(output_dir / f"{filename}_local_thresholding.png",
          (interface.astype('uint8') * 255))